# TODO 8.1：Colab 单候选结构化剪枝验证

这个 Notebook 用于验证 `structured_05`、`structured_10`、`structured_15`、`structured_20`。四个候选各有独立单元，**每次只执行其中一个**，避免同时保留四份 BF16 权重。

每个候选按以下顺序执行：真实结构剪枝 → 单 token 前向 → 落盘 → 独立重载 → 883 条 full-dev 评测 → 报告复制到 Google Drive → 可选删除 `/content` 中的大权重。只测试一个候选时不要执行项目的 `finalize`，因为质量拐点需要四档完整证据。

## 1. 连接 GPU 并挂载 Google Drive

先在 Colab 菜单选择“运行时 → 更改运行时类型 → GPU”。建议显存不少于约 22 GiB、系统内存不少于约 25 GiB、`/content` 可用空间不少于约 18 GiB。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import re
import shutil
import subprocess
from pathlib import Path

print(subprocess.run(['nvidia-smi'], text=True, capture_output=True, check=True).stdout)
print(subprocess.run(['free', '-h'], text=True, capture_output=True, check=True).stdout)
print(subprocess.run(['df', '-h', '/content'], text=True, capture_output=True, check=True).stdout)

query = subprocess.run(
    ['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits'],
    text=True, capture_output=True, check=True,
).stdout.strip().splitlines()[0]
vram_mib = int(re.findall(r'\d+', query)[0])
disk_free_gib = shutil.disk_usage('/content').free / 1024**3
memory_total_kib = int(next(line.split()[1] for line in Path('/proc/meminfo').read_text().splitlines() if line.startswith('MemTotal:')))
memory_total_gib = memory_total_kib / 1024**2
print(f'GPU memory: {vram_mib / 1024:.1f} GiB; RAM: {memory_total_gib:.1f} GiB; /content free: {disk_free_gib:.1f} GiB')
if vram_mib < 22 * 1024:
    raise RuntimeError('当前 GPU 显存不足以可靠运行 7B BF16 基线与剪枝评测，请申请更大显存的运行时。')
if memory_total_gib < 24:
    raise RuntimeError('当前系统内存不足 24 GiB，请选择高内存运行时。')
if disk_free_gib < 18:
    raise RuntimeError('/content 可用空间不足 18 GiB，请清理空间或更换运行时。')

## 2. 配置路径

修改下面两个 Drive 路径：`DRIVE_PROJECT` 是项目目录，`MODEL_PATH` 是已经合并 LoRA、包含 `config.json` 和权重分片的完整模型目录。项目会复制到 `/content`，但跳过原有 `out/`、模型、虚拟环境和数据集；数据集通过符号链接读取。

In [ ]:
# ===== 只需要修改这里 =====
DRIVE_PROJECT = Path('/content/drive/MyDrive/OptimizationSystem')
MODEL_PATH = Path('/content/drive/MyDrive/models/Qwen2.5-7B-Instruct-QLoRA-step724-merged-fp16')
# ==========================

WORKSPACE = Path('/content/OptimizationSystem')
RESULTS_DIR = Path('/content/drive/MyDrive/structured_pruning_results')
CLEANUP_AFTER_EACH_PLAN = True

assert DRIVE_PROJECT.is_dir(), f'项目目录不存在：{DRIVE_PROJECT}'
assert (DRIVE_PROJECT / 'scripts/run_structured_pruning.py').is_file(), 'Drive 项目不是包含最新 8.1 实现的版本'
assert (MODEL_PATH / 'config.json').is_file(), f'合并模型 config.json 不存在：{MODEL_PATH}'
assert (DRIVE_PROJECT / 'datasets_contextual').is_dir(), 'datasets_contextual 不存在'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Project:', DRIVE_PROJECT)
print('Model:', MODEL_PATH)
print('Results:', RESULTS_DIR)

In [ ]:
WORKSPACE.mkdir(parents=True, exist_ok=True)
subprocess.run([
    'rsync', '-a',
    '--exclude=.git', '--exclude=.venv', '--exclude=.idea',
    '--exclude=out', '--exclude=models', '--exclude=datasets_contextual',
    f'{DRIVE_PROJECT}/', f'{WORKSPACE}/',
], check=True)
dataset_link = WORKSPACE / 'datasets_contextual'
if dataset_link.is_symlink() or dataset_link.exists():
    if dataset_link.is_symlink() and dataset_link.resolve() == (DRIVE_PROJECT / 'datasets_contextual').resolve():
        pass
    else:
        raise RuntimeError(f'目标已存在且不是预期的数据集链接：{dataset_link}')
else:
    dataset_link.symlink_to(DRIVE_PROJECT / 'datasets_contextual', target_is_directory=True)
print('Workspace ready:', WORKSPACE)
print('Dataset link:', dataset_link, '->', dataset_link.resolve())

## 3. 安装并检查依赖

本实现要求支持显式 `head_dim` 的 `transformers>=5.18`。如果安装后 Colab 明确提示必须重启运行时，请重启后从第 1 节重新执行。

In [ ]:
subprocess.run([
    'python', '-m', 'pip', 'install', '-r', str(WORKSPACE / 'requirements-server.txt')
], check=True)

In [ ]:
import torch
import transformers

def version_key(value):
    return tuple(int(part) for part in re.findall(r'\d+', value))

print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert torch.cuda.is_available(), 'CUDA 不可用'
assert version_key(transformers.__version__) >= version_key('5.18.0'), 'transformers 版本过低'

## 4. 准备契约并生成/恢复同源 BF16 基线

第一次执行会运行 883 条 BF16 full-dev 基线并把报告保存到 Drive。新 Colab 会话如果发现 Drive 中已有基线报告，就直接恢复，避免重复评测。这个基线必须来自与剪枝相同的合并模型；更换模型后应删除 Drive 中的旧基线报告并重新运行。

In [ ]:
subprocess.run(['python', str(WORKSPACE / 'scripts/run_structured_pruning.py'), 'prepare'], cwd=WORKSPACE, check=True)

local_baseline = WORKSPACE / 'out/structured_pruning/baseline/comparison_report.json'
drive_baseline = RESULTS_DIR / 'baseline_comparison_report.json'
local_baseline.parent.mkdir(parents=True, exist_ok=True)
if drive_baseline.is_file():
    shutil.copy2(drive_baseline, local_baseline)
    print('Restored baseline:', drive_baseline)
else:
    subprocess.run([
        'python', str(WORKSPACE / 'scripts/run_structured_pruning.py'),
        'evaluate-baseline', '--model-path', str(MODEL_PATH), '--device', 'cuda:0',
    ], cwd=WORKSPACE, check=True)
    shutil.copy2(local_baseline, drive_baseline)
    print('Saved baseline:', drive_baseline)

## 5. 报告归档与空间清理函数

默认 `CLEANUP_AFTER_EACH_PLAN=True`：质量评测完成并把报告复制到 Drive 后，删除 `/content` 中该候选的大型 `.safetensors` 权重。这样保留审计证据但不保留可部署模型。如果你要下载或部署剪枝模型，把开头的该变量改为 `False`。

In [ ]:
def archive_plan(plan_id):
    destination = RESULTS_DIR / plan_id
    destination.mkdir(parents=True, exist_ok=True)
    sources = {
        'pruning_manifest.json': WORKSPACE / f'out/structured_pruning/exports/{plan_id}/pruning_manifest.json',
        'quality.json': WORKSPACE / f'out/structured_pruning/evidence/{plan_id}/quality.json',
        'quality.receipt.json': WORKSPACE / f'out/structured_pruning/evidence/{plan_id}/quality.receipt.json',
        'comparison_report.json': WORKSPACE / f'out/structured_pruning/runs/{plan_id}/full_dev/comparison_report.json',
    }
    for target_name, source in sources.items():
        if not source.is_file():
            raise FileNotFoundError(source)
        shutil.copy2(source, destination / target_name)
    print('Reports archived to:', destination)
    if CLEANUP_AFTER_EACH_PLAN:
        export_dir = WORKSPACE / f'out/structured_pruning/exports/{plan_id}'
        weights = sorted(export_dir.glob('*.safetensors'))
        released = sum(path.stat().st_size for path in weights) / 1024**3
        for path in weights:
            path.unlink()
        print(f'Deleted {len(weights)} temporary weight shard(s); released {released:.2f} GiB.')
        print('Manifest and quality evidence remain, but the deployable model weights are no longer retained.')

## 6A. 只执行本单元：5% 候选 `structured_05`

这是建议优先运行的最低风险候选。

In [ ]:
PLAN = 'structured_05'
subprocess.run(['python', str(WORKSPACE / 'scripts/run_structured_pruning.py'), 'export', '--plan-id', PLAN, '--model-path', str(MODEL_PATH), '--device', 'cuda:0'], cwd=WORKSPACE, check=True)
subprocess.run(['python', str(WORKSPACE / 'scripts/run_structured_pruning.py'), 'evaluate', '--plan-id', PLAN, '--device', 'cuda:0'], cwd=WORKSPACE, check=True)
archive_plan(PLAN)

## 6B. 只执行本单元：10% 候选 `structured_10`

In [ ]:
PLAN = 'structured_10'
subprocess.run(['python', str(WORKSPACE / 'scripts/run_structured_pruning.py'), 'export', '--plan-id', PLAN, '--model-path', str(MODEL_PATH), '--device', 'cuda:0'], cwd=WORKSPACE, check=True)
subprocess.run(['python', str(WORKSPACE / 'scripts/run_structured_pruning.py'), 'evaluate', '--plan-id', PLAN, '--device', 'cuda:0'], cwd=WORKSPACE, check=True)
archive_plan(PLAN)

## 6C. 只执行本单元：15% 候选 `structured_15`

In [ ]:
PLAN = 'structured_15'
subprocess.run(['python', str(WORKSPACE / 'scripts/run_structured_pruning.py'), 'export', '--plan-id', PLAN, '--model-path', str(MODEL_PATH), '--device', 'cuda:0'], cwd=WORKSPACE, check=True)
subprocess.run(['python', str(WORKSPACE / 'scripts/run_structured_pruning.py'), 'evaluate', '--plan-id', PLAN, '--device', 'cuda:0'], cwd=WORKSPACE, check=True)
archive_plan(PLAN)

## 6D. 只执行本单元：20% 候选 `structured_20`

In [ ]:
PLAN = 'structured_20'
subprocess.run(['python', str(WORKSPACE / 'scripts/run_structured_pruning.py'), 'export', '--plan-id', PLAN, '--model-path', str(MODEL_PATH), '--device', 'cuda:0'], cwd=WORKSPACE, check=True)
subprocess.run(['python', str(WORKSPACE / 'scripts/run_structured_pruning.py'), 'evaluate', '--plan-id', PLAN, '--device', 'cuda:0'], cwd=WORKSPACE, check=True)
archive_plan(PLAN)

## 7. 查看已保存结果

这个单元只读取 Drive 中已经完成的候选，不要求四档全部存在。

In [ ]:
import json

baseline_payload = json.loads((RESULTS_DIR / 'baseline_comparison_report.json').read_text())
baseline_row = next(row for row in baseline_payload['variants'] if row['variant'] == 'structured_json')
baseline_accuracy = baseline_row['metrics']['numeric_accuracy']
print(f'BF16 baseline accuracy: {baseline_accuracy:.6f}')
for plan in ('structured_05', 'structured_10', 'structured_15', 'structured_20'):
    quality_path = RESULTS_DIR / plan / 'quality.json'
    if not quality_path.is_file():
        print(f'{plan}: not run')
        continue
    payload = json.loads(quality_path.read_text())
    metrics = payload['metrics']
    drop = metrics['baseline_accuracy'] - metrics['accuracy']
    print(f"{plan}: accuracy={metrics['accuracy']:.6f}, drop={drop:.6f}, parse_rate={metrics.get('parse_rate')}")